# Geração UV-Vis — Avaliação por Experimentos Similares

Notebook de geração evolutiva com **dois algoritmos com papéis distintos e não conflitantes**:

## Arquitetura do pipeline

```
SEMENTES  ──►  TED Zhang–Shasha  ──►  Recombinação  ──►  Validação
               (par mais distante)    anel-ponte-anel     SMILES
                     │
                     ▼
               Moléculas novas (candidatas)
                     │
                     ▼
               Fitness UV-Vis + NNSim
               (calculado APÓS geração,
                não influencia a recombinação)
                     │
                     ▼
               Seleção por Torneio (AG)
               ──► SEMENTES da próxima geração
```

| Etapa | Responsável | O que decide |
|---|---|---|
| **Recombinação** | Zhang–Shasha TED | *com quem* cada semente se recombina — prioriza pares estruturalmente distantes |
| **Validação** | RDKit + aromatização | se o SMILES resultante é químicamente válido |
| **Avaliação** | Fitness UV-Vis + NNSim | pontuação de cada molécula nova |
| **Seleção** | Torneio (AG) | *quais* moléculas viram sementes da geração seguinte |

O **fitness e o NNSim não interferem no algoritmo de recombinação** — eles apenas controlam
a pressão seletiva sobre *quem sobrevive* para a próxima geração.

A **novidade de Liu & Liu (IJMS 27:5916, 2026)** aplicada aqui é usar uma biblioteca de
cromóforos UV-Vis experimentalmente conhecidos como referência para o NNSim (Estágio 4),
tornando a avaliação de performance comparável a experimentos da literatura.


## 1. Dependências


In [ ]:
import random, sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, QED, AllChem
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem.Scaffolds import MurckoScaffold
from scipy.stats import wasserstein_distance
from zss import Node, simple_distance

random.seed(42)
np.random.seed(42)

_SA_DISPONIVEL = False
_sascorer = None
try:
    from rdkit.Contrib.SA_Score import sascorer as _sascorer
    _SA_DISPONIVEL = True
except ImportError:
    try:
        from rdkit.Chem import RDConfig
        sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
        import sascorer as _sascorer
        _SA_DISPONIVEL = True
    except ImportError:
        print('SA score indisponivel — fixado em 5.0')

_params_pains = FilterCatalogParams()
_params_pains.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
CATALOGO_PAINS = FilterCatalog(_params_pains)

PALETA = {'azul':'#2a78d6','laranja':'#eb6834','verde':'#2aad6a',
          'roxo':'#8b44b5','grade':'#e1e0d9','eixo':'#c3c2b7',
          'primario':'#0b0b0b','secund':'#52514e'}

def estilizar(ax):
    for s in ['top','right']: ax.spines[s].set_visible(False)
    for s in ['left','bottom']: ax.spines[s].set_color(PALETA['eixo'])
    ax.tick_params(colors=PALETA['secund'])
    ax.xaxis.label.set_color(PALETA['secund'])
    ax.yaxis.label.set_color(PALETA['secund'])
    ax.title.set_color(PALETA['primario'])

print('Dependencias carregadas. SA score:', _SA_DISPONIVEL)


## 2. Base de moléculas


In [ ]:
CAMINHO_BASE = 'C://Users//Cliente//Documents//Git//gerador_smiles//data//banco_smiles_aromatizados.parquet'
df_base = pd.read_parquet(CAMINHO_BASE)
print(f'Moleculas na base: {len(df_base):,}')

canonicos_base_complexo = set(df_base['SMILES_Canonico_Complexo'].dropna())
canonicos_base_aromatico = set()
for smi in df_base['smiles'].dropna():
    mol = Chem.MolFromSmiles(str(smi))
    if mol is not None:
        canonicos_base_aromatico.add(Chem.MolToSmiles(mol))
print(f'SMILES aromaticos unicos na base: {len(canonicos_base_aromatico):,}')


## 3. Biblioteca de referência UV-Vis

Conjunto curado de **25 cromóforos** com absorção UV-Vis confirmada experimentalmente na literatura.
Organizados em 6 classes estruturais. Usados para:
- Calcular **NNSim** (Estágio 4) durante o fitness
- Comparar propriedades das moléculas geradas
- Medir **fragment recovery** das classes cromofóricas


In [ ]:
BIBLIOTECA_REFERENCIA = {
    # Azo (lambda_max experimental 330–500 nm)
    'azobenzene':        ('c1ccc(/N=N/c2ccccc2)cc1',             'azo'),
    'metil_laranja':     ('CN(C)c1ccc(/N=N/c2ccc(S(=O)(=O)O)cc2)cc1', 'azo'),
    'disp_orange_1':     ('O=[N+]([O-])c1ccc(/N=N/c2ccc(N)cc2)cc1', 'azo'),
    '4_OH_azo':          ('Oc1ccc(/N=N/c2ccccc2)cc1',           'azo'),
    'disp_red_1':        ('CCN(CCO)c1ccc(/N=N/c2ccc([N+](=O)[O-])cc2)cc1', 'azo'),

    # Antraquinona (lambda_max 250–700 nm)
    'antraquinona':      ('O=C1c2ccccc2C(=O)c2ccccc21',          'antraquinona'),
    'alizarina':         ('O=C1c2ccc(O)cc2C(=O)c2c(O)c(O)ccc21','antraquinona'),
    '1_4_diaminoantra':  ('Nc1ccc2C(=O)c3ccc(N)cc3C(=O)c2c1',   'antraquinona'),

    # Cumarina (lambda_max 310–450 nm)
    'cumarina':          ('O=c1ccc2ccccc2o1',                    'cumarina'),
    '7_amino_cumarina':  ('Nc1ccc2cc(=O)oc2c1',                 'cumarina'),
    'cumarina_1':        ('CCN(CC)c1ccc2cc(=O)oc2c1',           'cumarina'),

    # Estilbeno / push-pull (lambda_max 295–500 nm)
    'trans_estilbeno':   ('c1ccc(/C=C/c2ccccc2)cc1',            'estilbeno'),
    '4_nitro_stilb':     ('O=[N+]([O-])c1ccc(/C=C/c2ccccc2)cc1','estilbeno'),
    'DANS':              ('CN(C)c1ccc(/C=C/c2ccc([N+](=O)[O-])cc2)cc1', 'estilbeno'),

    # Indigo (lambda_max 600–680 nm)
    'indigo':            ('O=C1C(=C2NC(=O)c3ccccc23)Nc2ccccc21','indigo'),

    # Naftalimida (lambda_max 330–450 nm)
    'naftalimida':       ('O=C1NC(=O)c2cc3ccccc3cc21',          'naftalimida'),
    '4_amino_naftali':   ('Nc1ccc2C(=O)NC(=O)c2c1',            'naftalimida'),

    # Policíclico aromático (UV 250–380 nm)
    'antraceno':         ('c1ccc2cc3ccccc3cc2c1',               'policiclico'),
    'pireno':            ('c1cc2ccc3cccc4ccc(c1)c2c34',         'policiclico'),
    'fluorescina':       ('OC(=O)c1ccccc1-c1c2ccc(=O)cc2oc2cc(O)ccc12', 'policiclico'),

    # Push-pull simples (UV-vis)
    'DMABN':             ('N#Cc1ccc(N(C)C)cc1',                 'push_pull'),
    'p_nitroaniline':    ('Nc1ccc([N+](=O)[O-])cc1',            'push_pull'),
    '4_NMA':             ('CN(C)c1ccc(C=O)cc1',                 'push_pull'),
    'quinolina':         ('c1ccc2ncccc2c1',                     'push_pull'),
    'acridina':          ('c1ccc2nc3ccccc3cc2c1',               'push_pull'),
}

# Validar e calcular propriedades da biblioteca
registros_ref = []
fps_ref = []       # fingerprints pre-computados para NNSim
nomes_ref = []

for nome, (smi, classe) in BIBLIOTECA_REFERENCIA.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f'  AVISO: {nome} nao parseou, ignorado')
        continue
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fps_ref.append(fp)
    nomes_ref.append(nome)
    sa = _sascorer.calculateScore(mol) if _SA_DISPONIVEL else 5.0
    aneis = mol.GetRingInfo().AtomRings()
    n_arom = sum(1 for a in aneis if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in a))
    registros_ref.append({
        'nome': nome, 'classe': classe, 'smiles': smi,
        'mw':   round(Descriptors.MolWt(mol), 1),
        'logp': round(Descriptors.MolLogP(mol), 2),
        'tpsa': round(Descriptors.TPSA(mol), 1),
        'hbd':  Descriptors.NumHDonors(mol),
        'hba':  Descriptors.NumHAcceptors(mol),
        'rot':  Descriptors.NumRotatableBonds(mol),
        'qed':  round(QED.qed(mol), 3),
        'sa':   round(sa, 2),
        'n_aneis_arom': n_arom,
    })

df_ref = pd.DataFrame(registros_ref).set_index('nome')
print(f'Biblioteca de referencia: {len(df_ref)} cromoforos validos')
print('Distribuicao por classe:')
display(df_ref['classe'].value_counts().to_frame())
display(df_ref[['classe','mw','logp','qed','sa','n_aneis_arom']].round(2))


In [ ]:
# Propriedades da biblioteca de referencia vs. base de dados
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

props = [('mw', 'Peso Molecular (Da)', (100, 800)),
         ('logp','logP', (-5, 10)),
         ('n_aneis_arom', 'N. Aneis Aromaticos', (0, 8))]

for ax, (col, lbl, xlim) in zip(axes, props):
    vals_ref = df_ref[col].dropna().values
    ax.hist(vals_ref, bins=15, color=PALETA['laranja'], alpha=0.8,
            edgecolor='white', label='Referencia (25 mol.)')
    ax.axvline(vals_ref.mean(), color=PALETA['laranja'], lw=2, ls='--')
    ax.set_xlabel(lbl); ax.set_ylabel('Frequencia')
    ax.set_title(f'{lbl} — Referencia UV-Vis')
    ax.legend(frameon=False, fontsize=9)
    ax.set_xlim(*xlim)
    estilizar(ax)

plt.suptitle('Perfil de propriedades da biblioteca de referencia UV-Vis\n(cromoforos com absorcao experimental confirmada)',
             y=1.02, color=PALETA['primario'])
plt.tight_layout()
plt.show()


## 4. Catálogo de anéis e pontes (TED Zhang–Shasha)


In [ ]:
def extrair_catalogo_aneis(df, sufixo):
    colunas = {
        'base': f'SMILES_Base_Anel{sufixo}',
        'smiles_modificado': f'SMILES_Modificado_Anel{sufixo}',
        'frag1': f'Frag_Estrutura_1_Anel{sufixo}', 'pos1': f'Posicao_1_Anel{sufixo}',
        'frag2': f'Frag_Estrutura_2_Anel{sufixo}', 'pos2': f'Posicao_2_Anel{sufixo}',
        'frag3': f'Frag_Estrutura_3_Anel{sufixo}', 'pos3': f'Posicao_3_Anel{sufixo}',
    }
    sub = df[list(colunas.values())].copy()
    sub.columns = list(colunas.keys())
    sub['origem'] = f'Anel{sufixo}'
    return sub

catalogo_bruto = pd.concat(
    [extrair_catalogo_aneis(df_base, 1), extrair_catalogo_aneis(df_base, 2)],
    ignore_index=True)
catalogo_aneis = catalogo_bruto.drop_duplicates(subset='smiles_modificado').reset_index(drop=True)

catalogo_pontes = (df_base[['ID_Ponte','Ponte_Estrutura']].dropna().drop_duplicates()
                   .reset_index(drop=True))
catalogo_pontes = catalogo_pontes[
    catalogo_pontes['Ponte_Estrutura'].str.strip() != '#ERROR!'].reset_index(drop=True)
catalogo_pontes_lista = catalogo_pontes.to_dict('records')

def valor_valido(v): return pd.notna(v) and str(v).strip() not in {'', 'nan', 'None'}

def anel_para_arvore(row):
    raiz = Node(f"ANEL:{row['base']}")
    itens = [(str(row[f'pos{s}']), str(row[f'frag{s}'])) for s in (1,2,3)
             if valor_valido(row[f'frag{s}'])]
    for pos, frag in sorted(itens, key=lambda x: x[0]):
        no = Node(f'POS:{pos}')
        no.addkid(Node(f'FRAG:{frag}'))
        raiz.addkid(no)
    return raiz

def tamanho_arvore(no):
    return 1 + sum(tamanho_arvore(f) for f in no.children)

def ted_normalizada(a, b):
    d = simple_distance(a, b)
    return d / max(tamanho_arvore(a), tamanho_arvore(b), 1)

arvores_catalogo = [anel_para_arvore(row) for _, row in catalogo_aneis.iterrows()]
print(f'Aneis unicos: {len(catalogo_aneis):,}  |  Pontes: {len(catalogo_pontes)}'
      f'  |  Arvores TED: {len(arvores_catalogo):,}')


## 5. Recombinação e validação


In [ ]:
def substituir_numeracao_anel(smiles, de='1', para='2'):
    return smiles.replace(de, para)

def validar_e_canonizar(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None: return None, False
    return Chem.MolToSmiles(mol, canonical=True), True

def aromatizar_aneis_6_membros(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    for anel in mol.GetRingInfo().AtomRings():
        if len(anel) != 6: continue
        for k in range(len(anel)):
            at1, at2 = anel[k], anel[(k+1)%len(anel)]
            lig = mol.GetBondBetweenAtoms(at1, at2)
            if lig is None: return None
            lig.SetBondType(Chem.rdchem.BondType.AROMATIC)
        for idx in anel: mol.GetAtomWithIdx(idx).SetIsAromatic(True)
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return Chem.MolToSmiles(mol)

MAPA_COLUNAS = {
    'base_Anel1':'SMILES_Base_Anel1','smiles_modificado_Anel1':'SMILES_Modificado_Anel1',
    'frag1_Anel1':'Frag_Estrutura_1_Anel1','pos1_Anel1':'Posicao_1_Anel1',
    'frag2_Anel1':'Frag_Estrutura_2_Anel1','pos2_Anel1':'Posicao_2_Anel1',
    'frag3_Anel1':'Frag_Estrutura_3_Anel1','pos3_Anel1':'Posicao_3_Anel1',
    'base_Anel2':'SMILES_Base_Anel2','smiles_modificado_Anel2':'SMILES_Modificado_Anel2',
    'frag1_Anel2':'Frag_Estrutura_1_Anel2','pos1_Anel2':'Posicao_1_Anel2',
    'frag2_Anel2':'Frag_Estrutura_2_Anel2','pos2_Anel2':'Posicao_2_Anel2',
    'frag3_Anel2':'Frag_Estrutura_3_Anel2','pos3_Anel2':'Posicao_3_Anel2',
}
print('Funcoes de recombinacao e validacao definidas.')


## 6. Fitness UV-Vis com Avaliação por Experimentos Similares (Estágio 4)

A nova componente de fitness é o **NNSim** (Nearest-Neighbor Similarity) ao conjunto de
referência — conforme Estágio 4 de Liu & Liu (2026):
*'Similarity analysis against known active compounds'*.

Para corantes UV-Vis, a 'atividade' é a absorção UV-Vis; os 'compostos ativos conhecidos'
são os cromóforos da biblioteca de referência (Seção 3).

| Componente | Peso padrão | Estágio (Liu & Liu) |
|---|---|---|
| MW (200–700 Da)  | 0.07 | 2 |
| QED              | 0.06 | 2 |
| SA score         | 0.07 | 2 |
| π-conjugação     | 0.20 | UV-Vis |
| D-π-A            | 0.17 | UV-Vis |
| Planaridade      | 0.11 | UV-Vis |
| **NNSim (ref)**  | **0.32** | **4** |
| PAINS (penalidade) | −30% | 2 |


In [ ]:
_DOADORES = {
    'amino_primario':   Chem.MolFromSmarts('[NX3;H2]'),
    'amino_secundario': Chem.MolFromSmarts('[NX3;H1;!$(NC=O)]'),
    'amino_terciario':  Chem.MolFromSmarts('[NX3;H0;!$(NC=O);!$(N=*)]'),
    'hidroxil':         Chem.MolFromSmarts('[OX2H]'),
    'alcoxil':          Chem.MolFromSmarts('[OX2;!$(OC=O)][#6]'),
    'tiol':             Chem.MolFromSmarts('[SX2H]'),
    'tioeter':          Chem.MolFromSmarts('[SX2;!$(SC=O)][#6]'),
}
_ACEPTORES = {
    'nitro':    Chem.MolFromSmarts('[N+](=O)[O-]'),
    'nitroso':  Chem.MolFromSmarts('[NX2]=O'),
    'ciano':    Chem.MolFromSmarts('[CX2]#[NX1]'),
    'carbonil': Chem.MolFromSmarts('[CX3](=O)[#6]'),
    'sulfonil': Chem.MolFromSmarts('[SX4](=O)(=O)'),
    'imina':    Chem.MolFromSmarts('[CX3]=[NX2]'),
}

def _n_matches(mol, padroes):
    return sum(1 for p in padroes.values() if p and mol.HasSubstructMatch(p))


def calcular_nnsim(mol, fps_referencia):
    """Max Tanimoto (Morgan FP r=2) entre mol e a biblioteca de referencia."""
    if not fps_referencia:
        return 0.0
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    sims = DataStructs.BulkTanimotoSimilarity(fp, fps_referencia)
    return float(max(sims))


def calcular_nnsim_por_classe(mol, fps_referencia, nomes_ref, df_ref):
    """NNSim separado por classe de cromoforo."""
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    sims = DataStructs.BulkTanimotoSimilarity(fp, fps_referencia)
    por_classe = {}
    for nome, sim in zip(nomes_ref, sims):
        cls = df_ref.loc[nome, 'classe']
        por_classe[cls] = max(por_classe.get(cls, 0.0), float(sim))
    return por_classe


def calcular_fitness_exp(smiles, fps_referencia, pesos=None):
    """
    Fitness com NNSim (Estagio 4) + componentes estruturais UV-Vis (Estagio 2).
    pesos: dict com chaves mw, qed, sa, pi, d_a, planar, nnsim.
    """
    if pesos is None:
        pesos = {'mw':0.07,'qed':0.06,'sa':0.07,'pi':0.20,'d_a':0.17,'planar':0.11,'nnsim':0.32}

    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None

    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd  = Descriptors.NumHDonors(mol)
    hba  = Descriptors.NumHAcceptors(mol)
    rot  = Descriptors.NumRotatableBonds(mol)
    qed  = QED.qed(mol)
    sa   = _sascorer.calculateScore(mol) if _SA_DISPONIVEL else 5.0
    tem_pains = CATALOGO_PAINS.HasMatch(mol)

    aneis = mol.GetRingInfo().AtomRings()
    n_arom = sum(1 for a in aneis if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in a))
    n_d    = _n_matches(mol, _DOADORES)
    n_a    = _n_matches(mol, _ACEPTORES)

    mw_score    = 1.0 if 200<=mw<=700 else (mw/200 if mw<200 else max(0,(1-(mw-700)/300)))
    qed_score   = float(qed)
    sa_score    = max(0.0, 1.0 - (sa-1.0)/9.0)
    pi_score    = min(1.0, n_arom/3.0)
    d_a_score   = (min(1,n_d) + min(1,n_a)) / 2.0
    planar_score= max(0.0, 1.0 - rot/8.0)
    nnsim_raw   = calcular_nnsim(mol, fps_referencia)
    # Transformacao logistica suave: evita pressure excessiva para nnsim=1
    # nnsim=0.3 -> ~0.5; nnsim=0.5 -> ~0.73; nnsim=0.7 -> ~0.88
    nnsim_score = 1.0 / (1.0 + 2.718**(-8*(nnsim_raw - 0.3)))
    pains_f     = 0.70 if tem_pains else 1.0

    fitness = (
        pesos['mw']     * mw_score    +
        pesos['qed']    * qed_score   +
        pesos['sa']     * sa_score    +
        pesos['pi']     * pi_score    +
        pesos['d_a']    * d_a_score   +
        pesos['planar'] * planar_score +
        pesos['nnsim']  * nnsim_score
    ) * pains_f

    return {
        'mw':round(mw,2),'logp':round(logp,2),'tpsa':round(tpsa,2),
        'hbd':hbd,'hba':hba,'rot':rot,
        'qed':round(qed,3),'sa':round(sa,2),
        'n_aneis_arom':n_arom,'n_doadores':n_d,'n_aceptores':n_a,
        'tem_pains':tem_pains,
        'mw_score':round(mw_score,3),'qed_score':round(qed_score,3),
        'sa_score':round(sa_score,3),'pi_score':round(pi_score,3),
        'd_a_score':round(d_a_score,3),'planar_score':round(planar_score,3),
        'nnsim_raw':round(nnsim_raw,4),'nnsim_score':round(nnsim_score,4),
        'fitness':round(fitness,4),
    }


# Teste com cromoforos conhecidos
_testes_exp = {
    'azobenzene':     'c1ccc(/N=N/c2ccccc2)cc1',
    'cumarina_1':     'CCN(CC)c1ccc2cc(=O)oc2c1',
    'DANS':           'CN(C)c1ccc(/C=C/c2ccc([N+](=O)[O-])cc2)cc1',
    'benzeno':        'c1ccccc1',
}
print('Fitness experimental (com NNSim):')
for nome, smi in _testes_exp.items():
    f = calcular_fitness_exp(smi, fps_ref)
    if f: print(f'  {nome:20s}: fitness={f["fitness"]:.4f}  nnsim_raw={f["nnsim_raw"]:.4f}')


## 7. Seleção por Torneio (AG)


In [ ]:
def selecionar_sementes_torneio(df, n_sementes, col_fitness='fitness',
                                 k_torneio=3, rng=None):
    if rng is None: rng = random.Random(42)
    n = len(df)
    if n == 0: return df.copy()
    n_sementes = min(n_sementes, n)
    k = min(k_torneio, n)
    indices = []
    for _ in range(n_sementes):
        conc = rng.sample(range(n), k)
        indices.append(max(conc, key=lambda i: df.iloc[i][col_fitness]))
    return df.iloc[indices].reset_index(drop=True)

print('Selecao por torneio definida.')


## 8. Loop evolutivo com fitness baseado em experimentos similares

### Como Zhang–Shasha e fitness interagem

Dentro de cada geração, o fluxo é:

1. **TED Zhang–Shasha** compara a árvore da semente com as árvores de `n_sondas` candidatos
   do catálogo e **ordena por distância decrescente** (mais diverso primeiro).
2. Tenta recombinar com o parceiro mais distante. Se inválido, tenta o próximo na fila TED.
3. **Somente após** produzir uma molécula válida é calculado o `fitness` (incluindo NNSim).
4. O fitness vai para a função de **seleção por torneio**, que escolhe as sementes da
   próxima geração — *nunca retroage sobre a etapa de recombinação*.

### Hipóteses testadas: peso do NNSim vs. componentes estruturais

Cada configuração representa um balanço diferente entre atrair o espaço químico
para cromóforos conhecidos (NNSim) e favorecer critérios estruturais UV-Vis:

| Config | nnsim weight | Hipótese |
|---|---|---|
| `sem_referencia` | 0 % | Só critérios estruturais — igual ao notebook anterior |
| `nnsim_leve`     | 25% | Pequena atração por cromoforos conhecidos |
| `nnsim_moderado` | 40% | Equilíbrio referência / estrutura |
| `nnsim_forte`    | 55% | Forte pressão para análogos de referência |
| `uvvis_puro`     | 0 % | Maximiza π-conjugação e push-pull, ignora referência |
| `uniforme`       | 14% | Sem hipótese central, pesos iguais |

> **Em todas as configs**, o TED guia exatamente da mesma forma as recombinações.
> A única diferença entre configs é *quem vira semente* na próxima rodada.


In [ ]:
CONFIGS_PESOS_EXP = {
    'sem_referencia': {'mw':0.10,'qed':0.10,'sa':0.10,'pi':0.30,'d_a':0.25,'planar':0.15,'nnsim':0.00},
    'nnsim_leve':     {'mw':0.08,'qed':0.07,'sa':0.08,'pi':0.22,'d_a':0.18,'planar':0.12,'nnsim':0.25},
    'nnsim_moderado': {'mw':0.06,'qed':0.06,'sa':0.06,'pi':0.18,'d_a':0.14,'planar':0.10,'nnsim':0.40},
    'nnsim_forte':    {'mw':0.05,'qed':0.05,'sa':0.05,'pi':0.15,'d_a':0.10,'planar':0.05,'nnsim':0.55},
    'uvvis_puro':     {'mw':0.05,'qed':0.05,'sa':0.05,'pi':0.40,'d_a':0.35,'planar':0.10,'nnsim':0.00},
    'uniforme':       {'mw':1/7, 'qed':1/7, 'sa':1/7, 'pi':1/7, 'd_a':1/7, 'planar':1/7,'nnsim':1/7},
}
for n,p in CONFIGS_PESOS_EXP.items():
    s = sum(p.values())
    assert abs(s-1.0)<1e-9, f'{n}: soma={s:.4f}'
print('Configuracoes definidas:')
display(pd.DataFrame(CONFIGS_PESOS_EXP).T.round(3))


In [ ]:
# ── Funcao de uma geracao (com fitness experimental) ────────────────────────
def rodar_geracao_exp(
    df_sementes, df_historico, catalogo_pontes_lista,
    can_c_acum, can_a_acum, numero_geracao,
    fps_ref, pesos_cfg,
    n_sondas=5, semente=44, comb_inv=None,
):
    if comb_inv is None: comb_inv = set()
    cat_bruto = pd.concat(
        [extrair_catalogo_aneis(df_historico, 1),
         extrair_catalogo_aneis(df_historico, 2)], ignore_index=True)
    cat_aneis = cat_bruto.drop_duplicates(subset='smiles_modificado').reset_index(drop=True)
    arvores   = [anel_para_arvore(r) for _, r in cat_aneis.iterrows()]
    n_cat     = len(cat_aneis)

    base1 = extrair_catalogo_aneis(df_sementes, 1)
    rng_p = random.Random(semente)
    rng_b = random.Random(semente*7+1)
    n_ef  = min(n_sondas, n_cat)

    vistos, linhas, cans = set(), [], []
    rem_inv = rem_dup = rem_ex = tent = 0

    for pos in range(len(base1)):
        row_i = base1.iloc[pos]
        arv_i = anel_para_arvore(row_i)
        sondas = rng_p.sample(range(n_cat), n_ef)
        dists  = sorted([(idx, ted_normalizada(arv_i, arvores[idx])) for idx in sondas],
                        key=lambda p: p[1], reverse=True)
        for melhor_idx, dist in dists:
            tent += 1
            row_j = cat_aneis.iloc[melhor_idx]
            ponte = rng_b.choice(catalogo_pontes_lista)
            smi_i = row_i['smiles_modificado']
            smi_j = substituir_numeracao_anel(row_j['smiles_modificado'])
            assin = (smi_i, ponte['Ponte_Estrutura'], smi_j)
            if assin in comb_inv: rem_inv+=1; continue
            smiles_c = f"{smi_i}{ponte['Ponte_Estrutura']}{smi_j}"
            can, ok  = validar_e_canonizar(smiles_c)
            if not ok: comb_inv.add(assin); rem_inv+=1; continue
            if can in vistos: rem_dup+=1; continue
            if can in can_c_acum: rem_ex+=1; continue
            vistos.add(can); cans.append(can)
            linhas.append({
                'base_Anel1':row_i['base'],'smiles_modificado_Anel1':row_i['smiles_modificado'],
                'frag1_Anel1':row_i['frag1'],'pos1_Anel1':row_i['pos1'],
                'frag2_Anel1':row_i['frag2'],'pos2_Anel1':row_i['pos2'],
                'frag3_Anel1':row_i['frag3'],'pos3_Anel1':row_i['pos3'],
                'ID_Ponte':ponte['ID_Ponte'],'Ponte_Estrutura':ponte['Ponte_Estrutura'],
                'base_Anel2':row_j['base'],'smiles_modificado_Anel2':row_j['smiles_modificado'],
                'frag1_Anel2':row_j['frag1'],'pos1_Anel2':row_j['pos1'],
                'frag2_Anel2':row_j['frag2'],'pos2_Anel2':row_j['pos2'],
                'frag3_Anel2':row_j['frag3'],'pos3_Anel2':row_j['pos3'],
                'distancia_ted':dist,'SMILES_Complexo':smiles_c,
            })
            break

    df_fa = pd.DataFrame(linhas)
    if not df_fa.empty:
        df_fa['SMILES_Can_Candidato'] = cans
        df_fa['SMILES_Arom'] = df_fa['SMILES_Complexo'].apply(aromatizar_aneis_6_membros)
        df_fb = df_fa.dropna(subset=['SMILES_Arom']).reset_index(drop=True)
    else:
        df_fa['SMILES_Can_Candidato'] = pd.Series(dtype=object)
        df_fa['SMILES_Arom'] = pd.Series(dtype=object)
        df_fb = df_fa.copy()

    vistos2, linhas2 = set(), []
    for _, row in df_fb.iterrows():
        mol = Chem.MolFromSmiles(row['SMILES_Arom'])
        if mol is None: continue
        cf = Chem.MolToSmiles(mol)
        if cf in vistos2 or cf in can_a_acum: continue
        vistos2.add(cf); linhas2.append(row)

    df_ger = pd.DataFrame(linhas2).reset_index(drop=True)
    if not df_ger.empty:
        df_ger['smiles'] = df_ger['SMILES_Arom']
        recs_fit = []
        for smi in df_ger['smiles']:
            f = calcular_fitness_exp(smi, fps_ref, pesos_cfg)
            recs_fit.append(f if f else {'fitness':0.0,'nnsim_raw':0.0})
        df_fit = pd.DataFrame(recs_fit)
        for col in df_fit.columns: df_ger[col] = df_fit[col].values
    else:
        df_ger = pd.DataFrame(columns=['smiles','fitness','nnsim_raw'])
    df_ger['Geracao'] = numero_geracao
    return df_ger

print('Funcao rodar_geracao_exp definida.')


## 9. Execução multi-configuração


In [ ]:
MAX_GERACOES    = 12
N_SONDAS        = 6
N_SEMENTES      = 60
K_TORNEIO       = 3
N_SEED_INICIAL  = 400

df_base_seed = df_base.sample(N_SEED_INICIAL, random_state=42).reset_index(drop=True)

_can_c0 = set(df_base_seed['SMILES_Canonico_Complexo'].dropna())
_can_a0 = set()
for smi in df_base_seed['smiles'].dropna():
    mol = Chem.MolFromSmiles(str(smi))
    if mol: _can_a0.add(Chem.MolToSmiles(mol))

resultados = {}   # config -> DataFrame com todas as moleculas geradas
evolucao   = {}   # config -> lista de (geracao, n_novas, fit_med, nnsim_med)

for nome_cfg, pesos_cfg in CONFIGS_PESOS_EXP.items():
    hist     = df_base_seed.copy()
    can_c    = set(_can_c0)
    can_a    = set(_can_a0)
    seeds    = df_base_seed.copy()
    comb_inv = set()
    rng      = random.Random(42)
    frames   = []
    trajetoria = []

    for g in range(1, MAX_GERACOES+1):
        df_nova = rodar_geracao_exp(
            seeds, hist, catalogo_pontes_lista, can_c, can_a,
            numero_geracao=g, fps_ref=fps_ref, pesos_cfg=pesos_cfg,
            n_sondas=N_SONDAS, semente=42+g, comb_inv=comb_inv,
        )
        if len(df_nova) == 0: break
        frames.append(df_nova)
        trajetoria.append({
            'geracao': g,
            'n_novas': len(df_nova),
            'fitness_medio': df_nova['fitness'].mean() if 'fitness' in df_nova.columns else float('nan'),
            'nnsim_medio':   df_nova['nnsim_raw'].mean() if 'nnsim_raw' in df_nova.columns else float('nan'),
        })
        can_c |= set(df_nova.get('SMILES_Can_Candidato', pd.Series()).dropna())
        can_a |= set(df_nova['smiles'].dropna())
        df_nova_ren = df_nova.rename(columns=MAPA_COLUNAS)
        hist = pd.concat([hist, df_nova_ren], ignore_index=True)
        n_sel = min(N_SEMENTES, len(df_nova))
        seeds = selecionar_sementes_torneio(df_nova_ren, n_sel, 'fitness', K_TORNEIO, rng)

    df_run = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    resultados[nome_cfg] = df_run
    evolucao[nome_cfg]   = trajetoria
    n_mol = len(df_run)
    nn_m  = df_run['nnsim_raw'].mean() if not df_run.empty and 'nnsim_raw' in df_run.columns else float('nan')
    fit_m = df_run['fitness'].mean()    if not df_run.empty and 'fitness' in df_run.columns else float('nan')
    print(f'  {nome_cfg:18s}: {n_mol:4d} mol  fitness={fit_m:.4f}  nnsim_raw={nn_m:.4f}')

print('\nTodos os mini-loops concluidos.')


## 10. Trajetória evolutiva por configuração


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cores_cfg = plt.cm.tab10(np.linspace(0, 0.9, len(CONFIGS_PESOS_EXP)))

for (nome_cfg, traj), cor in zip(evolucao.items(), cores_cfg):
    if not traj: continue
    df_t = pd.DataFrame(traj)
    axes[0].plot(df_t['geracao'], df_t['fitness_medio'], marker='o',
                 color=cor, label=nome_cfg)
    axes[1].plot(df_t['geracao'], df_t['nnsim_medio'], marker='s',
                 color=cor, label=nome_cfg)

for ax, titulo, ylabel in [
    (axes[0], 'Evolucao do fitness medio por geracao', 'Fitness medio (0-1)'),
    (axes[1], 'Evolucao do NNSim medio por geracao', 'NNSim medio a referencia'),
]:
    ax.set_xlabel('Geracao')
    ax.set_ylabel(ylabel)
    ax.set_title(titulo)
    ax.legend(frameon=False, fontsize=8, ncol=2)
    ax.grid(axis='y', color=PALETA['grade'])
    ax.set_axisbelow(True)
    estilizar(ax)

plt.tight_layout()
plt.show()


## 11. Avaliação Estágios 3 e 4 — Liu & Liu (IJMS 2026)

### Estágio 3 (Novidade e Diversidade)
- **IntDiv1**: 1 − Tanimoto médio sobre pares (diversidade interna)
- **Scaffold Novelty**: scaffolds Bemis–Murcko ausentes na base de referência

### Estágio 4 (Relevância — adaptado UV-Vis)
- **NNSim médio**: similaridade média ao cromóforo mais próximo na biblioteca
- **Zona de análogos** (NNSim ≥ 0.40): % de moléculas na faixa de similaridade de análogos
- **Zona de vizinhos** (NNSim ≥ 0.25): espaço químico próximo a cromóforos conhecidos
- **Distância de Wasserstein** das propriedades (MW, logP) vs. biblioteca de referência


In [ ]:
def intdiv1(fps, n_amostra=400, rng_seed=42):
    rng = random.Random(rng_seed)
    n = len(fps)
    if n < 2: return float('nan')
    idx = list(range(n))
    if n > n_amostra: idx = rng.sample(idx, n_amostra)
    total, npares = 0.0, 0
    for i in range(len(idx)):
        par = [fps[idx[j]] for j in range(i+1, len(idx))]
        if par:
            total += sum(DataStructs.BulkTanimotoSimilarity(fps[idx[i]], par))
            npares += len(par)
    return round(1 - total/npares, 4) if npares else float('nan')

def scaffold_novelty_vs_base(smiles_list, can_a_base):
    scaffolds_base = set()
    for smi in list(can_a_base)[:3000]:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            try: scaffolds_base.add(Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol)))
            except: pass
    n_nov = n_tot = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None: continue
        try: s = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except: continue
        n_tot += 1
        if s not in scaffolds_base: n_nov += 1
    return round(n_nov/n_tot, 4) if n_tot else 0.0

print('Funcoes Stage 3+4 definidas. Calculando avaliacoes...')

linhas_aval = []
for nome_cfg, df_run in resultados.items():
    if df_run.empty:
        linhas_aval.append({'config': nome_cfg}); continue

    smiles_list = df_run['smiles'].dropna().tolist()
    fps_run = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s),2,nBits=2048)
               for s in smiles_list if Chem.MolFromSmiles(s) is not None]

    # Stage 3
    div1      = intdiv1(fps_run)
    scaf_nov  = scaffold_novelty_vs_base(smiles_list, _can_a0)
    novelty   = sum(1 for s in smiles_list if s not in _can_a0) / len(smiles_list) if smiles_list else 0.0

    # Stage 4 — NNSim
    nnsim_vals = df_run['nnsim_raw'].dropna().values if 'nnsim_raw' in df_run.columns else np.array([])
    nnsim_med   = float(np.mean(nnsim_vals)) if len(nnsim_vals) else float('nan')
    nnsim_max   = float(np.max(nnsim_vals))  if len(nnsim_vals) else float('nan')
    zona_anal   = float(np.mean(nnsim_vals >= 0.40)) if len(nnsim_vals) else float('nan')
    zona_viz    = float(np.mean(nnsim_vals >= 0.25)) if len(nnsim_vals) else float('nan')

    # Stage 4 — Distancia de Wasserstein nas propriedades vs. referencia
    w_mw   = wasserstein_distance(df_run['mw'].dropna().values,   df_ref['mw'].values)   if 'mw'   in df_run.columns else float('nan')
    w_logp = wasserstein_distance(df_run['logp'].dropna().values, df_ref['logp'].values) if 'logp' in df_run.columns else float('nan')

    # Stage 2
    fit_med = df_run['fitness'].mean() if 'fitness' in df_run.columns else float('nan')
    qed_med = df_run['qed'].mean()     if 'qed'     in df_run.columns else float('nan')
    sa_med  = df_run['sa'].mean()      if 'sa'      in df_run.columns else float('nan')

    linhas_aval.append({
        'config': nome_cfg,
        'n_moleculas': len(smiles_list),
        'fitness_medio': round(fit_med, 4),
        # Stage 3
        'intdiv1':        div1,
        'scaffold_novelty': scaf_nov,
        'novelty_exata':  round(novelty, 4),
        # Stage 4
        'nnsim_medio':    round(nnsim_med, 4),
        'nnsim_max':      round(nnsim_max, 4),
        'zona_analogos':  round(zona_anal, 3),
        'zona_vizinhos':  round(zona_viz, 3),
        'wass_mw':        round(w_mw, 1),
        'wass_logp':      round(w_logp, 2),
        # Stage 2
        'qed_medio': round(qed_med, 3),
        'sa_medio':  round(sa_med, 2),
    })
    print(f'  {nome_cfg:18s}: IntDiv1={div1:.4f} | NNSim={nnsim_med:.4f}'
          f' | zona_anal={zona_anal:.2%} | W_MW={w_mw:.1f}')

df_aval = pd.DataFrame(linhas_aval).set_index('config')
print('\n=== TABELA DE AVALIACAO (Estagios 2, 3, 4 — Liu & Liu 2026) ===')
display(df_aval.sort_values('nnsim_medio', ascending=False))


## 12. Visualizações comparativas (Estágios 3 e 4)


In [ ]:
configs_ord = df_aval['nnsim_medio'].sort_values(ascending=False).index.tolist()

# ── Painel 1: 6 métricas em barras ──────────────────────────────────────────
metricas_vis = [
    ('intdiv1',       'IntDiv1 (Stage 3)',         PALETA['azul']),
    ('scaffold_novelty','Scaffold Novelty (S3)',    PALETA['verde']),
    ('nnsim_medio',   'NNSim medio (Stage 4)',      PALETA['laranja']),
    ('zona_analogos', 'Zona Analogos >0.40 (S4)',  PALETA['roxo']),
    ('zona_vizinhos', 'Zona Vizinhos >0.25 (S4)',  PALETA['azul']),
    ('novelty_exata', 'Novelty Exata (Stage 3)',    PALETA['verde']),
]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, (col, titulo, cor) in zip(axes.flatten(), metricas_vis):
    vals = [df_aval.loc[c, col] if c in df_aval.index and not pd.isna(df_aval.loc[c,col]) else 0
            for c in configs_ord]
    bars = ax.bar(configs_ord, vals, color=cor, width=0.55, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                f'{v:.3f}', ha='center', fontsize=9, color=PALETA['primario'])
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.18+1e-6)
    ax.grid(axis='y', color=PALETA['grade'])
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', rotation=25)
    estilizar(ax)

plt.suptitle('Avaliacao Estagios 3 e 4 — Liu & Liu (IJMS 27:5916, 2026)\n'
             'Maior NNSim = mais analogos a cromoforos com atividade UV-Vis experimental',
             y=1.01, color=PALETA['primario'])
plt.tight_layout()
plt.show()

# ── Painel 2: distribuicao de NNSim por configuracao ─────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 5))
for (nome_cfg, df_run), cor in zip(resultados.items(), cores_cfg):
    if df_run.empty or 'nnsim_raw' not in df_run.columns: continue
    vals = df_run['nnsim_raw'].dropna().values
    ax2.hist(vals, bins=30, alpha=0.5, color=cor, label=nome_cfg, density=True)

ax2.axvline(0.25, color='gray', ls='--', lw=1.2, label='Limiar vizinho (0.25)')
ax2.axvline(0.40, color='black', ls='--', lw=1.5, label='Limiar analogo (0.40)')
ax2.set_xlabel('NNSim maximo a cromoforo de referencia')
ax2.set_ylabel('Densidade')
ax2.set_title('Distribuicao de NNSim por configuracao\n'
              '(Estagio 4 — Relevancia por similaridade a cromoforos conhecidos)')
ax2.legend(frameon=False, fontsize=9, ncol=2)
ax2.grid(axis='y', color=PALETA['grade'])
ax2.set_axisbelow(True)
estilizar(ax2)
plt.tight_layout()
plt.show()

# ── Painel 3: Wasserstein MW e logP (alinhamento a referencia) ────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, col, lbl in [(axes3[0],'wass_mw','Wasserstein MW'),
                      (axes3[1],'wass_logp','Wasserstein logP')]:
    vals = [df_aval.loc[c, col] if c in df_aval.index else float('nan')
            for c in configs_ord]
    bars = ax.bar(configs_ord, vals, color=PALETA['azul'], alpha=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                f'{v:.2f}', ha='center', fontsize=9)
    ax.set_title(f'{lbl} vs. biblioteca de referencia\n(menor = mais similar a cromoforos reais)')
    ax.set_ylabel(lbl)
    ax.grid(axis='y', color=PALETA['grade'])
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', rotation=25)
    estilizar(ax)
plt.tight_layout()
plt.show()

# Resumo
melhor_nnsim = df_aval['nnsim_medio'].idxmax()
melhor_anal  = df_aval['zona_analogos'].idxmax()
melhor_div   = df_aval['intdiv1'].idxmax()
melhor_w_mw  = df_aval['wass_mw'].idxmin()
print('─'*65)
print('VENCEDOR POR CRITERIO (Estagios 3+4, Liu & Liu 2026)')
print(f'  Maior NNSim medio:       {melhor_nnsim}  ({df_aval.loc[melhor_nnsim,"nnsim_medio"]:.4f})')
print(f'  Mais analogos (>0.40):   {melhor_anal}  ({df_aval.loc[melhor_anal,"zona_analogos"]:.2%})')
print(f'  Maior IntDiv1:           {melhor_div}  ({df_aval.loc[melhor_div,"intdiv1"]:.4f})')
print(f'  MW mais similar a ref.:  {melhor_w_mw}  (W={df_aval.loc[melhor_w_mw,"wass_mw"]:.1f})')
print('─'*65)


## 13. Melhores candidatos — analogos a cromóforos de referência

Top moléculas geradas classificadas por NNSim à referência e por classe.


In [ ]:
# Pool de todos os resultados
frames_pool = []
for nome_cfg, df_run in resultados.items():
    if df_run.empty: continue
    sub = df_run.copy()
    sub['config_origem'] = nome_cfg
    frames_pool.append(sub)

df_pool = pd.concat(frames_pool, ignore_index=True) if frames_pool else pd.DataFrame()

if not df_pool.empty and 'nnsim_raw' in df_pool.columns:
    # Melhor por classe de referencia
    print('=== TOP 3 CANDIDATOS POR CLASSE DE CROMOFORO REFERENCIA ===')
    for nome_ref, (smi_ref, classe) in BIBLIOTECA_REFERENCIA.items():
        mol_ref = Chem.MolFromSmiles(smi_ref)
        if mol_ref is None: continue
        fp_ref_i = AllChem.GetMorganFingerprintAsBitVect(mol_ref, 2, nBits=2048)

        sims_i = []
        for smi in df_pool['smiles'].dropna():
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None: sims_i.append(0.0); continue
            fp_i = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
            sims_i.append(DataStructs.TanimotoSimilarity(fp_ref_i, fp_i))
        df_pool[f'sim_{nome_ref}'] = sims_i

    # Top 5 mais similares a cada referencia (sem duplicatas)
    cols_show = ['smiles','nnsim_raw','fitness','n_aneis_arom','n_doadores','n_aceptores','config_origem']
    cols_show = [c for c in cols_show if c in df_pool.columns]

    top_global = (df_pool.sort_values('nnsim_raw', ascending=False)
                  .drop_duplicates(subset='smiles').head(20))
    print('\n=== TOP 20 CANDIDATOS GLOBAIS (maior NNSim a qualquer cromoforo) ===')
    display(top_global[cols_show].reset_index(drop=True))

    # Distribuicao de NNSim no pool completo
    print(f'\nPool total: {len(df_pool):,} moleculas')
    nnsim_all = df_pool['nnsim_raw'].dropna()
    print(f'NNSim medio: {nnsim_all.mean():.4f}  max: {nnsim_all.max():.4f}')
    print(f'Zona analogos (>=0.40): {(nnsim_all>=0.40).sum():,} ({(nnsim_all>=0.40).mean():.1%})')
    print(f'Zona vizinhos (>=0.25): {(nnsim_all>=0.25).sum():,} ({(nnsim_all>=0.25).mean():.1%})')
else:
    print('Pool vazio ou sem coluna nnsim_raw.')


## 14. Exportar resultados


In [ ]:
from pathlib import Path
SAIDA = Path('data')
SAIDA.mkdir(exist_ok=True)

# Avaliacao comparativa
arq_aval = SAIDA / 'avaliacao_experimental_uvvis.parquet'
df_aval.reset_index().to_parquet(arq_aval, index=False)
print(f'Avaliacao salva: {arq_aval}')

# Resultados por configuracao
for nome_cfg, df_run in resultados.items():
    if df_run.empty: continue
    arq = SAIDA / f'geradas_exp_{nome_cfg}.parquet'
    df_run.to_parquet(arq, index=False)
    print(f'  {nome_cfg}: {len(df_run):,} moleculas -> {arq.name}')

# Top candidatos
if not df_pool.empty and 'nnsim_raw' in df_pool.columns:
    top_cands = (df_pool.sort_values('nnsim_raw', ascending=False)
                 .drop_duplicates(subset='smiles').head(50))
    arq_top = SAIDA / 'top_candidatos_exp_uvvis.parquet'
    top_cands.to_parquet(arq_top, index=False)
    print(f'Top-50 candidatos: {arq_top}')


## 15. Conclusão

Este notebook implementa uma avaliação baseada em **experimentos similares** (Liu & Liu, IJMS 27:5916, 2026):

### Estágio 4 — O que é novo
- **Biblioteca de referência**: 25 cromóforos UV-Vis com atividade experimental confirmada
  (azos, antraquinonas, coumarinas, estilbenos, índigo, naftalimidas, push-pull)
- **NNSim** como componente do fitness: aproxima a pressão seletiva para o espaço químico
  de cromóforos conhecidos
- **Zona de análogos** (NNSim ≥ 0.40): fracão das moléculas geradas que são análogos
  estruturais de cromóforos experimentais
- **Distância de Wasserstein**: quão alinhadas estão as distribuições de MW/logP com a referência

### Interpretação (Stage 4, Liu & Liu)
> *'A molecule predicted to be active is not the same as an experimentally active molecule'
> — mas uma molécula similar a cromóforos experimentais tem maior plausibilidade de absorver UV-Vis.*

Configurações com maior peso de NNSim produzem **mais análogos** de cromóforos reais (zona de análogos
mais ampla, maior NNSim médio), porém podem reduzir diversidade estrutural (IntDiv1 menor). A
configuração ideal depende do objetivo: **exploração** (sem_referencia/uvvis_puro) vs.
**aproximação a cores experimentais** (nnsim_moderado/nnsim_forte).


In [ ]:
if not df_aval.empty:
    cols_sum = [c for c in ['n_moleculas','fitness_medio','intdiv1','nnsim_medio',
                             'zona_analogos','zona_vizinhos','wass_mw','qed_medio'] if c in df_aval.columns]
    print('RESUMO FINAL — Avaliacao por experimentos similares')
    print('='*70)
    display(df_aval[cols_sum].sort_values('nnsim_medio', ascending=False))
    print('='*70)
    print(f'Melhor NNSim medio:  {melhor_nnsim}')
    print(f'Maior zona analogos: {melhor_anal}')
    print(f'Maior diversidade:   {melhor_div}')
